In [1]:
import pandas as pd
import json
import zipfile

In [2]:
def load_fda_zip(zip_filename):
    """
    Loads an openFDA ZIP file and returns
    the results as a pandas DataFrame.
    """

    with zipfile.ZipFile(zip_filename, "r") as z:

        json_filename = z.namelist()[0]

        with z.open(json_filename) as f:
            data = json.load(f)

    df = pd.json_normalize(data["results"])

    return df

In [4]:
shortages = load_fda_zip(
    "drug-shortages-0001-of-0001.json.zip"
)

ndc = load_fda_zip(
    "drug-ndc-0001-of-0001.json.zip"
)

recalls = load_fda_zip(
    "drug-enforcement-0001-of-0001.json.zip"
)

print("Shortages:", shortages.shape)
print("NDC:", ndc.shape)
print("Recalls:", recalls.shape)

Shortages: (1601, 36)
NDC: (138217, 32)
Recalls: (17975, 45)


In [5]:
package_rows = []

for _, row in ndc.iterrows():

    packages = row.get("packaging")

    if not isinstance(packages, list):
        continue

    for package in packages:

        if not isinstance(package, dict):
            continue

        package_rows.append({
            "product_ndc": row.get("product_ndc"),
            "package_ndc": package.get("package_ndc"),
            "package_description": package.get("description"),
            "generic_name_ndc": row.get("generic_name"),
            "brand_name_ndc": row.get("brand_name"),
            "labeler_name": row.get("labeler_name"),
            "dosage_form_ndc": row.get("dosage_form"),
            "marketing_category": row.get("marketing_category")
        })

packages = pd.DataFrame(package_rows)

print("Package records:", len(packages))

packages.head()

Package records: 256387


,product_ndc,package_ndc,package_description,generic_name_ndc,brand_name_ndc,labeler_name,dosage_form_ndc,marketing_category
0,71288-432,71288-432-81,10 SYRINGE in 1 CARTON (71288-432-81) / .3 mL...,Enoxaparin Sodium,Enoxaparin Sodium,Meitheal Pharmaceuticals Inc.,INJECTION,ANDA
1,71288-440,71288-440-02,10 VIAL in 1 CARTON (71288-440-02) / 1 mL in ...,Desmopressin acetate,Desmopressin acetate,Meitheal Pharmaceuticals Inc.,INJECTION,ANDA
2,71288-441,71288-441-10,1 VIAL in 1 CARTON (71288-441-10) / 10 mL in ...,Desmopressin acetate,Desmopressin acetate,Meitheal Pharmaceuticals Inc.,INJECTION,ANDA
3,0409-0183,0409-0183-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0409-0183-01)...",GEMCITABINE,Gemcitabine,"Hospira, Inc.","INJECTION, SOLUTION",NDA
4,71288-450,71288-450-02,"25 VIAL, MULTI-DOSE in 1 CARTON (71288-450-02)...",Heparin Sodium,Heparin Sodium,Meitheal Pharmaceuticals Inc.,INJECTION,ANDA


In [6]:
shortage_ndcs = set(
    shortages["package_ndc"]
    .dropna()
    .astype(str)
    .str.strip()
)

directory_ndcs = set(
    packages["package_ndc"]
    .dropna()
    .astype(str)
    .str.strip()
)

matched_ndcs = shortage_ndcs.intersection(directory_ndcs)

print("Unique shortage NDCs:", len(shortage_ndcs))
print("Matched NDCs:", len(matched_ndcs))

match_rate = (
    len(matched_ndcs)
    / len(shortage_ndcs)
    * 100
)

print(f"NDC match rate: {match_rate:.2f}%")

Unique shortage NDCs: 1570
Matched NDCs: 1393
NDC match rate: 88.73%


In [7]:
unmatched_ndcs = shortage_ndcs - directory_ndcs

print("Unmatched shortage NDCs:", len(unmatched_ndcs))

list(unmatched_ndcs)[:20]

Unmatched shortage NDCs: 177


['58468-0127-1',
 '0641-6059-01',
 '69489-721-05',
 '0409-4910-34',
 '0409-2253-01',
 '63323-750-10',
 '10122-100-10',
 '0990-7139-09',
 '71288-722-32',
 '0338-9549-50',
 '13668-353-30',
 '0955-1735-30',
 '0338-9553-50',
 '63481-627-70',
 '0409-7535-01',
 '0378-4615-01',
 '62037-720-05',
 '0641-6024-10',
 '0002-4312-61',
 '59212-001-02']

In [8]:
shortages["package_ndc"] = (
    shortages["package_ndc"]
    .astype(str)
    .str.strip()
)

packages["package_ndc"] = (
    packages["package_ndc"]
    .astype(str)
    .str.strip()
)

shortage_enriched = shortages.merge(
    packages,
    on="package_ndc",
    how="left"
)

print("Original shortage rows:", len(shortages))
print("Merged rows:", len(shortage_enriched))

shortage_enriched.head()

Original shortage rows: 1601
Merged rows: 1601


,update_type,initial_posting_date,package_ndc,generic_name,contact_info,availability,update_date,therapeutic_category,dosage_form,presentation,...,change_date,openfda.pharm_class_moa,resolved_note,product_ndc,package_description,generic_name_ndc,brand_name_ndc,labeler_name,dosage_form_ndc,marketing_category
0,Reverified,04/02/2020,0409-2596-05,Midazolam Hydrochloride Injection,844-646-4398,Available,09/22/2026,"[Anesthesia, Neurology]",Injection,"Midazolam Hydrochloride, Injection, 50 mg/10 m...",...,NaN,NaN,NaN,0409-2596,"10 VIAL, MULTI-DOSE in 1 CARTON (0409-2596-05)...",MIDAZOLAM HYDROCHLORIDE,Midazolam,"Hospira, Inc.","INJECTION, SOLUTION",ANDA
1,Reverified,01/19/2023,62135-484-60,Quinapril Hydrochloride Tablet,845-268-5000 x 510 or Chartwell Customer servi...,Available,09/15/2026,[Cardiovascular],Tablet,"Quinapril Hydrochloride, Tablet, 5 mg (NDC 621...",...,NaN,NaN,NaN,62135-484,"60 TABLET, FILM COATED in 1 BOTTLE (62135-484-60)",Quinapril,Quinapril,"Chartwell RX, LLC","TABLET, FILM COATED",ANDA
2,Revised,04/07/2020,23155-473-42,Furosemide Injection,855-228-9470,Unavailable,09/01/2026,[Cardiovascular],Injection,"Furosemide, Injection, 10 mg/1 mL (NDC 23155-4...",...,NaN,NaN,NaN,23155-473,25 VIAL in 1 CARTON (23155-473-42) / 4 mL in ...,Furosemide,Furosemide,Heritage Pharmaceuticals Inc. d/b/a Avet Pharm...,"INJECTION, SOLUTION",ANDA
3,Reverified,03/23/2018,63323-286-20,Ropivacaine Hydrochloride Injection,888-386-1300,Unavailable,09/15/2026,[Anesthesia],Injection,"Naropin, Injection, 5 mg/1 mL (NDC 63323-286-20)",...,NaN,NaN,NaN,63323-286,5 AMPULE in 1 BOX (63323-286-20) / 20 mL in 1...,ROPIVACAINE HYDROCHLORIDE,Naropin,"Fresenius Kabi USA, LLC","INJECTION, SOLUTION",NDA
4,New,02/25/2026,0009-0225-02,Clindamycin Hydrochloride Capsule,844-646-4398,NaN,02/25/2026,[Anti-Infective],Capsule,"Cleocin Hydrochloride, Capsule, 150 mg (NDC 00...",...,NaN,NaN,NaN,0009-0225,100 CAPSULE in 1 BOTTLE (0009-0225-02),clindamycin hydrochloride,Cleocin Hydrochloride,Pharmacia & Upjohn Company LLC,CAPSULE,NDA


In [9]:
unmatched_shortages = shortages[
    ~shortages["package_ndc"].isin(directory_ndcs)
].copy()

print("Unmatched shortage rows:", len(unmatched_shortages))

unmatched_shortages[
    [
        "package_ndc",
        "generic_name",
        "company_name",
        "status",
        "update_date"
    ]
].head(20)

Unmatched shortage rows: 180


,package_ndc,generic_name,company_name,status,update_date
8,0603-3718-34,Frovatriptan Succinate Tablet,"Endo Pharmaceuticals, Inc.",To Be Discontinued,11/12/2025
20,76045-0212-10,Dexamethasone Sodium Phosphate Injection,"Fresenius Kabi USA, LLC",Current,09/15/2026
32,0409-4904-34,Lidocaine Hydrochloride Injection,"Hospira, Inc., a Pfizer Company",To Be Discontinued,11/07/2025
37,69918-501-05,Desmopressin Acetate Spray,Ferring,Current,05/01/2026
39,0025-2742-31,"Disopyramide Phosphate Capsule, Extended Release",Pfizer Inc.,To Be Discontinued,08/24/2026
44,0641-0121-25,Hydromorphone Hydrochloride Injection,"Hikma Pharmaceuticals USA, Inc.",Current,09/18/2026
46,0338-9545-50,Clindamycin Phosphate Injection,Baxter Healthcare,Current,09/17/2026
58,0409-1893-01,Morphine Sulfate Injection,"Hospira, Inc., a Pfizer Company",Current,09/22/2026
59,0574-4022-35,Bacitracin Ophthalmic Ointment,Padagis US LLC,Current,06/01/2026
67,58468-0123-1,Doxercalciferol Injection,Sanofi-Aventis U.S. LLC,To Be Discontinued,12/10/2025


In [10]:
print(
    unmatched_shortages["package_ndc"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

package_ndc
12    178
13      2
Name: count, dtype: int64


In [11]:
unmatched_shortages["package_ndc"].dropna().head(30).tolist()

['0603-3718-34',
 '76045-0212-10',
 '0409-4904-34',
 '69918-501-05',
 '0025-2742-31',
 '0641-0121-25',
 '0338-9545-50',
 '0409-1893-01',
 '0574-4022-35',
 '58468-0123-1',
 '0409-1530-01',
 '0409-5092-16',
 '13668-354-01',
 '59212-111-14',
 '0074-7098-03',
 '0378-0006-85',
 '13668-356-90',
 '49884-128-91',
 '0338-9553-50',
 '49884-563-11',
 '25021-230-05',
 '58468-0127-1',
 '0074-0522-60',
 '0409-7535-01',
 '17478-050-05',
 '13668-356-05',
 '63323-750-20',
 '0603-5165-32',
 '63323-671-20',
 '0053-6871-00']

In [13]:
for ndc in unmatched_shortages["package_ndc"].dropna().head(30):
    print(ndc)
    length_counts = (
    unmatched_shortages["package_ndc"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

print(length_counts.to_string())

0603-3718-34
76045-0212-10
0409-4904-34
69918-501-05
0025-2742-31
0641-0121-25
0338-9545-50
0409-1893-01
0574-4022-35
58468-0123-1
0409-1530-01
0409-5092-16
13668-354-01
59212-111-14
0074-7098-03
0378-0006-85
13668-356-90
49884-128-91
0338-9553-50
49884-563-11
25021-230-05
58468-0127-1
0074-0522-60
0409-7535-01
17478-050-05
13668-356-05
63323-750-20
0603-5165-32
63323-671-20
0053-6871-00
package_ndc
12    178
13      2


In [14]:
for column in recalls.columns:
    print(column)

status
city
state
country
classification
product_type
event_id
recalling_firm
address_1
address_2
postal_code
voluntary_mandated
initial_firm_notification
distribution_pattern
recall_number
product_description
product_quantity
reason_for_recall
recall_initiation_date
report_date
code_info
center_classification_date
openfda.application_number
openfda.brand_name
openfda.generic_name
openfda.manufacturer_name
openfda.product_ndc
openfda.product_type
openfda.route
openfda.substance_name
openfda.rxcui
openfda.spl_id
openfda.spl_set_id
openfda.package_ndc
openfda.is_original_packager
openfda.upc
openfda.nui
openfda.pharm_class_moa
openfda.pharm_class_epc
openfda.unii
termination_date
more_code_info
openfda.pharm_class_cs
openfda.pharm_class_pe
openfda.original_packager_product_ndc


In [15]:
recalls.head()

,status,city,state,country,classification,product_type,event_id,recalling_firm,address_1,address_2,...,openfda.upc,openfda.nui,openfda.pharm_class_moa,openfda.pharm_class_epc,openfda.unii,termination_date,more_code_info,openfda.pharm_class_cs,openfda.pharm_class_pe,openfda.original_packager_product_ndc
0,Ongoing,Woburn,MA,United States,Not Yet Classified,Drugs,99666,"Safecor Health, LLC",317 New Boston St,N/A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ongoing,"Sinnar, Nashik District",N/A,India,Class II,Drugs,81707,"Mylan Laboratories Limited, (Nashik FDF)","F-4 F-12, Malegaon M.I.D.C., Sinnar",N/A,...,"[0303781722933, 0303781723930, 0303781721936, ...","[N0000000070, N0000175561]",[Angiotensin 2 Receptor Antagonists [MoA]],[Angiotensin 2 Receptor Blocker [EPC]],"[80M03YXJ7I, 864V2Q084H]",NaN,NaN,NaN,NaN,NaN
2,Terminated,Sellersville,PA,United States,Class III,Drugs,64644,"Teva Pharmaceuticals USA, Inc.",650 Cathill Rd,N/A,...,NaN,"[N0000175825, N0000000100]",[Estrogen Receptor Agonists [MoA]],[Estrogen [EPC]],"[9S44LIC7OJ, 423D2T571U]",20140410,NaN,NaN,NaN,NaN
3,Ongoing,Ledgewood,NJ,United States,Class III,Drugs,99756,ImprimisRx NJ LLC,1705 Us Highway 46 Ste 4,,...,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN
4,Ongoing,Deerfield,IL,United States,Class II,Drugs,99500,Baxter Healthcare Corporation,1 Baxter Pkwy,N/A,...,NaN,NaN,NaN,NaN,[976728SY6Z],NaN,,NaN,NaN,NaN


In [16]:
print("Recall columns:", len(recalls.columns))
print("Recall rows:", len(recalls))

Recall columns: 45
Recall rows: 17975


In [17]:
for column in recalls.columns:
    if (
        "ndc" in column.lower()
        or "product" in column.lower()
        or "openfda" in column.lower()
    ):
        print(column)

product_type
product_description
product_quantity
openfda.application_number
openfda.brand_name
openfda.generic_name
openfda.manufacturer_name
openfda.product_ndc
openfda.product_type
openfda.route
openfda.substance_name
openfda.rxcui
openfda.spl_id
openfda.spl_set_id
openfda.package_ndc
openfda.is_original_packager
openfda.upc
openfda.nui
openfda.pharm_class_moa
openfda.pharm_class_epc
openfda.unii
openfda.pharm_class_cs
openfda.pharm_class_pe
openfda.original_packager_product_ndc


In [18]:
print("Package NDC field example:")
print(recalls["openfda.package_ndc"].dropna().head(10))

print("\nProduct NDC field example:")
print(recalls["openfda.product_ndc"].dropna().head(10))

Package NDC field example:
1     [0378-1721-93, 0378-1722-93, 0378-1723-93, 037...
2     [0555-9025-42, 0555-9026-58, 0555-9027-42, 055...
4                          [43066-150-10, 43066-360-20]
5     [0032-1203-70, 0032-1203-64, 0032-1206-07, 003...
7     [50111-560-01, 50111-560-02, 50111-560-03, 501...
8     [65862-854-30, 65862-855-30, 65862-856-30, 658...
9     [27241-255-01, 27241-254-01, 27241-253-01, 272...
11                         [0338-9640-12, 0338-9647-12]
13    [0338-9644-01, 0338-9644-12, 0338-9646-01, 033...
14    [49230-206-92, 49230-206-94, 49230-206-95, 492...
Name: openfda.package_ndc, dtype: object

Product NDC field example:
1          [0378-1721, 0378-1722, 0378-1723, 0378-1724]
2          [0555-9025, 0555-9026, 0555-9027, 0555-9028]
4                                [43066-150, 43066-360]
5     [0032-0045, 0032-1203, 0032-1206, 0032-1212, 0...
7                     [50111-450, 50111-560, 50111-561]
8          [65862-854, 65862-855, 65862-856, 65862-857]
9       

In [19]:
print(
    "Recall records with package NDC:",
    recalls["openfda.package_ndc"].notna().sum()
)

print(
    "Recall records with product NDC:",
    recalls["openfda.product_ndc"].notna().sum()
)

print(
    "Total recall records:",
    len(recalls)
)

Recall records with package NDC: 3236
Recall records with product NDC: 3236
Total recall records: 17975


In [20]:
package_example = recalls["openfda.package_ndc"].dropna().iloc[0]
product_example = recalls["openfda.product_ndc"].dropna().iloc[0]

print("Package NDC type:", type(package_example))
print("Product NDC type:", type(product_example))

Package NDC type: <class 'list'>
Product NDC type: <class 'list'>


In [21]:
recall_packages = recalls[
    [
        "event_id",
        "recall_number",
        "recalling_firm",
        "classification",
        "reason_for_recall",
        "recall_initiation_date",
        "openfda.package_ndc"
    ]
].copy()

recall_packages = recall_packages.explode(
    "openfda.package_ndc"
)

recall_packages = recall_packages.rename(
    columns={
        "openfda.package_ndc": "package_ndc"
    }
)

recall_packages["package_ndc"] = (
    recall_packages["package_ndc"]
    .astype("string")
    .str.strip()
)

recall_packages.head(10)

,event_id,recall_number,recalling_firm,classification,reason_for_recall,recall_initiation_date,package_ndc
0,99666,N/A,"Safecor Health, LLC",Not Yet Classified,Cross Contamination with Other Products,20260814,<NA>
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1721-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1722-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1723-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1724-93
2,64644,D-682-2013,"Teva Pharmaceuticals USA, Inc.",Class III,Failed Impurity/Degradation Specification; an ...,20130315,0555-9025-42
2,64644,D-682-2013,"Teva Pharmaceuticals USA, Inc.",Class III,Failed Impurity/Degradation Specification; an ...,20130315,0555-9026-58
2,64644,D-682-2013,"Teva Pharmaceuticals USA, Inc.",Class III,Failed Impurity/Degradation Specification; an ...,20130315,0555-9027-42
2,64644,D-682-2013,"Teva Pharmaceuticals USA, Inc.",Class III,Failed Impurity/Degradation Specification; an ...,20130315,0555-9028-58
3,99756,D-0835-2026,ImprimisRx NJ LLC,Class III,Subpotent Drug,20260824,<NA>


In [22]:
print("Original recall records:", len(recalls))
print("Recall-package relationships:", len(recall_packages))

print(
    "Unique recall package NDCs:",
    recall_packages["package_ndc"].nunique()
)

recall_ndcs = set(
    recall_packages["package_ndc"].dropna()
)

shortage_ndcs = set(
    shortages["package_ndc"].dropna()
)

overlap = recall_ndcs.intersection(shortage_ndcs)

print(
    "NDCs in BOTH recalls and shortages:",
    len(overlap)
)

Original recall records: 17975
Recall-package relationships: 40316
Unique recall package NDCs: 11233
NDCs in BOTH recalls and shortages: 435
